# Multi-Skill Graphic Prep

This notebook creates Venn diagrams for both attacking and defensive players:

## Attacking Players
- **≥5 goals**, **≥3 assists**, or **>35 successful dribbles**

For each category, copy relevant player headshots into a staging folder as circular PNGs and export coordinates for the final canvas.

## Defensive Players
- **≥30 interceptions**, **≥150 ball recoveries**, or **≥40 clearances**

In [137]:
# Imports, paths, and global configuration
from pathlib import Path
from collections import defaultdict
import shutil

import pandas as pd
from PIL import Image, ImageOps, ImageDraw

LEAGUE = 'cg'  # 'cg' = Montenegro 1. CFL | 'srb' = Serbian SuperLiga

DATA_DIR = Path("..") / "data"
RAW_MATCH_DIR = DATA_DIR / "raw" / LEAGUE / "raw_by_match"
PROCESSED_DIR = DATA_DIR / "processed" / LEAGUE
PHOTOS_DIR = PROCESSED_DIR / "player_photos"

ASSET_DIR = Path("..") / "outputs" / LEAGUE / "final_posts" / "multi_skill_assets"
ASSET_DIR.mkdir(parents=True, exist_ok=True)
PHOTO_EXPORT_DIR = ASSET_DIR / "player_headshots"
PHOTO_EXPORT_DIR.mkdir(exist_ok=True)
EXPORT_PHOTO_SIZE = 520

# Attacking thresholds
GOAL_THRESHOLD = 5
ASSIST_THRESHOLD = 3
DRIBBLE_THRESHOLD = 35

# Defensive thresholds
INTERCEPTION_THRESHOLD = 40
RECOVERY_THRESHOLD = 200
CLEARANCE_THRESHOLD = 55

---
# Attacking Players Analysis

In [138]:
# Build attacking player dataset
def photo_path_for_player(pid):
    try:
        pid_int = int(pid)
    except (TypeError, ValueError):
        return None
    photo_path = PHOTOS_DIR / f"{pid_int}.png"
    return photo_path if photo_path.exists() else None

# Load player season statistics
players = pd.read_csv(PROCESSED_DIR / "player_season_statistics.csv")
players["display_name"] = players["shortName"].fillna(players["name"])
players["photo_path"] = players["player_id"].apply(photo_path_for_player)

# Fill missing attacking stats
players["successful_dribbles"] = players["dribbles_successful"].fillna(0)

# Apply attacking thresholds
players["goal_flag"] = players["goals"] >= GOAL_THRESHOLD
players["assist_flag"] = players["assists"] >= ASSIST_THRESHOLD
players["dribble_flag"] = players["successful_dribbles"] > DRIBBLE_THRESHOLD

# Filter to qualified attacking players
qualified_attacking = players.loc[players[["goal_flag", "assist_flag", "dribble_flag"]].any(axis=1)].copy()
qualified_attacking = qualified_attacking.sort_values(["goal_flag", "assist_flag", "dribble_flag", "successful_dribbles"], 
                                   ascending=[False, False, False, False]).reset_index(drop=True)

print(f"Qualified attacking players: {len(qualified_attacking)}")
qualified_attacking[["display_name", "position", "team_name", "goals", "assists", "successful_dribbles"]]

Qualified attacking players: 25


,display_name,position,team_name,goals,assists,successful_dribbles
0,Ž. Popović,M,OFK Petrovac,5.0,4.0,37.0
1,B. Sekulić,F,FK Mornar Bar,8.0,4.0,17.0
2,S. Denković,M,FK Mornar Bar,5.0,1.0,66.0
3,B. Mrvaljević,M,Mladost DG,5.0,2.0,42.0
4,A. Ražnatović,D,FK Sutjeska Nikšić,5.0,2.0,18.0
5,A. Kajević,M,FK Dečić Tuzi,5.0,1.0,15.0
6,A. Bašić,F,OFK Petrovac,6.0,1.0,9.0
7,Đ. Maksimović,F,FK Jezero,9.0,0.0,8.0
8,O. Rolović,F,FK Jezero,5.0,1.0,2.0
9,A. Bošnjak,M,FK Arsenal Tivat,4.0,3.0,63.0


In [139]:
# Export attacking player headshots and placement table
def save_circular_headshot(source_path, dest_path, size=EXPORT_PHOTO_SIZE):
    img = Image.open(source_path).convert("RGBA")
    img = ImageOps.fit(img, (size, size), Image.LANCZOS)
    mask = Image.new("L", (size, size), 0)
    ImageDraw.Draw(mask).ellipse((0, 0, size, size), fill=255)
    img.putalpha(mask)
    img.save(dest_path)

flag_map = {"G": "goal_flag", "A": "assist_flag", "D": "dribble_flag"}
segments_attacking = defaultdict(list)
for _, row in qualified_attacking.iterrows():
    membership = tuple(sorted([key for key, col in flag_map.items() if row[col]]))
    if membership:
        segments_attacking[membership].append(row)

for key in segments_attacking:
    segments_attacking[key].sort(key=lambda r: r["successful_dribbles"], reverse=True)

# Refresh the export directory
ATTACKING_PHOTO_DIR = ASSET_DIR / "attacking_headshots"
ATTACKING_PHOTO_DIR.mkdir(exist_ok=True)
for existing in ATTACKING_PHOTO_DIR.glob("*"):
    if existing.is_file():
        existing.unlink()

copied_attacking = []
player_rows_attacking = []
for combo, entries in segments_attacking.items():
    combo_label = "|".join(combo)
    for entry in entries:
        player_rows_attacking.append(
            {
                "player_id": int(entry["player_id"]),
                "display_name": entry["display_name"],
                "position": entry["position"],
                "team_name": entry["team_name"],
                "combo": combo_label,
                "goals": int(entry["goals"]),
                "assists": int(entry["assists"]),
                "successful_dribbles": int(entry["successful_dribbles"]),
            }
        )
        source_path = entry["photo_path"]
        if source_path is None:
            continue
        dest_name = f"{int(entry['player_id'])}_{entry['display_name'].replace(' ', '_')}.png"
        dest_path = ATTACKING_PHOTO_DIR / dest_name
        save_circular_headshot(source_path, dest_path)
        copied_attacking.append(dest_path.name)

attacking_positions_df = pd.DataFrame(player_rows_attacking)
attacking_positions_csv = ASSET_DIR / "attacking_player_positions.csv"
attacking_positions_df.to_csv(attacking_positions_csv, index=False)

print(f"Copied {len(copied_attacking)} attacking player headshots into {ATTACKING_PHOTO_DIR}")
print(f"Placement table saved to {attacking_positions_csv}")
attacking_positions_df

Copied 24 attacking player headshots into ..\outputs\final_posts\multi_skill_assets\attacking_headshots
Placement table saved to ..\outputs\final_posts\multi_skill_assets\attacking_player_positions.csv


,player_id,display_name,position,team_name,combo,goals,assists,successful_dribbles
0,1030468,Ž. Popović,M,OFK Petrovac,A|D|G,5,4,37
1,856483,B. Sekulić,F,FK Mornar Bar,A|G,8,4,17
2,95583,S. Denković,M,FK Mornar Bar,D|G,5,1,66
3,1149289,B. Mrvaljević,M,Mladost DG,D|G,5,2,42
4,1000770,A. Ražnatović,D,FK Sutjeska Nikšić,G,5,2,18
5,76273,A. Kajević,M,FK Dečić Tuzi,G,5,1,15
6,791606,A. Bašić,F,OFK Petrovac,G,6,1,9
7,905970,Đ. Maksimović,F,FK Jezero,G,9,0,8
8,196634,O. Rolović,F,FK Jezero,G,5,1,2
9,808566,A. Bošnjak,M,FK Arsenal Tivat,A|D,4,3,63


In [140]:
# Build defensive player dataset
players_def = pd.read_csv(PROCESSED_DIR / "player_season_statistics.csv")
players_def["display_name"] = players_def["shortName"].fillna(players_def["name"])
players_def["photo_path"] = players_def["player_id"].apply(photo_path_for_player)

# Fill missing defensive stats
players_def["interceptions"] = players_def["interceptions"].fillna(0)
players_def["ball_recoveries"] = players_def["ball_recoveries"].fillna(0)
players_def["clearances"] = players_def["clearances"].fillna(0)

# Apply defensive thresholds
players_def["interception_flag"] = players_def["interceptions"] >= INTERCEPTION_THRESHOLD
players_def["recovery_flag"] = players_def["ball_recoveries"] > RECOVERY_THRESHOLD
players_def["clearance_flag"] = players_def["clearances"] > CLEARANCE_THRESHOLD

# Filter to qualified defensive players
qualified_defensive = players_def.loc[players_def[["interception_flag", "recovery_flag", "clearance_flag"]].any(axis=1)].copy()
qualified_defensive = qualified_defensive.sort_values(["interception_flag", "recovery_flag", "clearance_flag", "ball_recoveries"], 
                                   ascending=[False, False, False, False]).reset_index(drop=True)

print(f"Qualified defensive players: {len(qualified_defensive)}")
qualified_defensive[["display_name", "position", "team_name", "interceptions", "ball_recoveries", "clearances"]]

Qualified defensive players: 17


,display_name,position,team_name,interceptions,ball_recoveries,clearances
0,N. Jovičić,D,FK Jezero,52.0,262.0,79.0
1,L. Ujkaj,D,FK Dečić Tuzi,46.0,233.0,67.0
2,L. Uskoković,D,Mladost DG,41.0,215.0,55.0
3,A. Šćekić,M,FK Sutjeska Nikšić,42.0,215.0,27.0
4,A. Čepić,M,FK Jedinstvo Bijelo Polje,50.0,213.0,7.0
5,N. Jovićević,D,FK Jezero,40.0,206.0,54.0
6,I. Martinović,D,FK Mornar Bar,45.0,193.0,62.0
7,B. Kopitović,D,FK Sutjeska Nikšić,42.0,165.0,39.0
8,N. A. Cordoba,M,Mladost DG,40.0,161.0,17.0
9,O. Obradović,D,FK Jedinstvo Bijelo Polje,23.0,243.0,65.0


In [141]:
# Export defensive player headshots and placement table
flag_map_def = {"I": "interception_flag", "R": "recovery_flag", "C": "clearance_flag"}
segments_defensive = defaultdict(list)
for _, row in qualified_defensive.iterrows():
    membership = tuple(sorted([key for key, col in flag_map_def.items() if row[col]]))
    if membership:
        segments_defensive[membership].append(row)

for key in segments_defensive:
    segments_defensive[key].sort(key=lambda r: r["ball_recoveries"], reverse=True)

# Refresh the export directory
DEFENSIVE_PHOTO_DIR = ASSET_DIR / "defensive_headshots"
DEFENSIVE_PHOTO_DIR.mkdir(exist_ok=True)
for existing in DEFENSIVE_PHOTO_DIR.glob("*"):
    if existing.is_file():
        existing.unlink()

copied_defensive = []
player_rows_defensive = []
for combo, entries in segments_defensive.items():
    combo_label = "|".join(combo)
    for entry in entries:
        player_rows_defensive.append(
            {
                "player_id": int(entry["player_id"]),
                "display_name": entry["display_name"],
                "position": entry["position"],
                "team_name": entry["team_name"],
                "combo": combo_label,
                "interceptions": int(entry["interceptions"]),
                "ball_recoveries": int(entry["ball_recoveries"]),
                "clearances": int(entry["clearances"]),
            }
        )
        source_path = entry["photo_path"]
        if source_path is None:
            continue
        dest_name = f"{int(entry['player_id'])}_{entry['display_name'].replace(' ', '_')}.png"
        dest_path = DEFENSIVE_PHOTO_DIR / dest_name
        save_circular_headshot(source_path, dest_path)
        copied_defensive.append(dest_path.name)

defensive_positions_df = pd.DataFrame(player_rows_defensive)
defensive_positions_csv = ASSET_DIR / "defensive_player_positions.csv"
defensive_positions_df.to_csv(defensive_positions_csv, index=False)

print(f"Copied {len(copied_defensive)} defensive player headshots into {DEFENSIVE_PHOTO_DIR}")
print(f"Placement table saved to {defensive_positions_csv}")
defensive_positions_df

Copied 16 defensive player headshots into ..\outputs\final_posts\multi_skill_assets\defensive_headshots
Placement table saved to ..\outputs\final_posts\multi_skill_assets\defensive_player_positions.csv


,player_id,display_name,position,team_name,combo,interceptions,ball_recoveries,clearances
0,1182552,N. Jovičić,D,FK Jezero,C|I|R,52,262,79
1,1063206,L. Ujkaj,D,FK Dečić Tuzi,C|I|R,46,233,67
2,605370,L. Uskoković,D,Mladost DG,I|R,41,215,55
3,836566,A. Šćekić,M,FK Sutjeska Nikšić,I|R,42,215,27
4,1185130,A. Čepić,M,FK Jedinstvo Bijelo Polje,I|R,50,213,7
5,1035787,N. Jovićević,D,FK Jezero,I|R,40,206,54
6,286837,I. Martinović,D,FK Mornar Bar,C|I,45,193,62
7,138937,B. Kopitović,D,FK Sutjeska Nikšić,I,42,165,39
8,1163021,N. A. Cordoba,M,Mladost DG,I,40,161,17
9,960227,O. Obradović,D,FK Jedinstvo Bijelo Polje,C|R,23,243,65


In [ ]:
# ─── Instagram Post Composition (Venn diagrams) ────────────
from PIL import Image

ASSETS_DIR = Path("..") / "assets"
FIGURES_DIR = Path("..") / "outputs" / LEAGUE / "figures"
FINAL_DIR = Path("..") / "outputs" / LEAGUE / "final_posts"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
INSTAGRAM_SIZE = (1080, 1350)
SAFE_ZONE = {'left': 30, 'top': 250, 'right': 1050, 'bottom': 1270}

for fig_name in ["venn_atacking.png", "venn_defensive.png"]:
    chart_path = FIGURES_DIR / fig_name
    if not chart_path.exists():
        print(f"Skipping {fig_name} (not found)")
        continue

    background = Image.open(ASSETS_DIR / 'background.png').convert('RGBA')
    if background.size != INSTAGRAM_SIZE:
        background = background.resize(INSTAGRAM_SIZE, Image.Resampling.LANCZOS)

    chart = Image.open(chart_path).convert('RGBA')
    safe_w = SAFE_ZONE['right'] - SAFE_ZONE['left']
    safe_h = SAFE_ZONE['bottom'] - SAFE_ZONE['top']
    chart = chart.resize((safe_w, safe_h), Image.Resampling.LANCZOS)
    background.paste(chart, (SAFE_ZONE['left'], SAFE_ZONE['top']), chart)

    out_name = fig_name.replace('.png', '_instagram.png')
    out_path = FINAL_DIR / out_name
    background.convert('RGB').save(out_path, quality=95)
    print(f"Saved: {out_path}")

